# PhoCUS Bilateral Acoustic Lens: Free-Field & Transcranial Simulation Pipeline

Runs the full k-Wave simulation pipeline for the PhoCUS bilateral holographic lens
(Methods Ch. 2): virtual-source phase extraction, phase-to-thickness lens design, lens
CAD/STL export, and forward-propagation lens-verification -- for both the free-field and
transcranial (skull + lens) configurations in a single run (2 heavy `kspaceFirstOrder3DC`
calls per medium, 4 total). Every artifact lands in one shared `OUTPUT_DIR`, with filenames
suffixed `_freefield` / `_skull` so nothing collides between the two media.

## Key physical differences between the two media
- `medium_skull` (Estrada 2016 / Jimenez 2022 bone constants) is only used in skull mode;
  free-field mode uses `medium_free` (pure water) for the same two simulate() calls.
- CFL is lower for the skull branch (heterogeneous, high sound-speed contrast: bone vs water)
  than for the free-field branch, at both the phase-extraction and verification steps.
- The lens's `LENS_Z_START` leaves 1 voxel of clearance from the transducer plane in both
  media (Section 8.5).
- Section 8.5's combined medium starts from pure water (free field) vs. from the skull's own
  density/speed/alpha maps (skull) before the lens is stamped on top.

## Notebook map

This notebook has its own section numbering independent of the thesis. Each heading below names its own notebook section and, in parentheses, the thesis section it
implements.

| # | Notebook section | Thesis section |
|---|---|---|
| 1 | Configuration & run parameters | Methods Sec. 2.4.1 (Table 2.2) |
| 1.1 | Excitation waveform options | Methods Sec. 2.6.1 |
| 2 | Skull acquisition, segmentation, acoustic properties | Methods Sec. 2.1 |
| 3 | Hippocampal target localisation | Methods Sec. 2.2 |
| 4 | Transducer geometry | Methods Sec. 2.4.1 |
| 5 | Simulation configurations -- shared execution options | Methods Sec. 2.4.2 |
| 6 | Virtual-source time reversal -- lens-standoff design targets | Methods Sec. 2.6.1 |
| 7 | Shared analysis & visualisation utilities | supports Results Ch. 3 |
| 8 | Main pipeline (8.1-8.11) | Methods Sec. 2.4.2, 2.6, 2.7; Results Ch. 3 |
| 9 | Run the pipeline -- select medium(s) | -- |

## Configuration switches: one variable, several methods
Several sections below are controlled by a single string/boolean variable that selects
between *alternative methods described in the thesis* -- change the variable and the
corresponding thesis alternative runs instead, with everything downstream adapting
automatically:

| Variable | Options | Selects between (thesis ref.) |
|---|---|---|
| `SOURCE_WAVEFORM` | `"toneburst"` / `"morlet"` / `"sinusoid"` | excitation waveform, Sec. 2.6.1 |
| `PHASE_EXTRACTION_METHOD` | `"kwave"` / `"manual"` | k-Wave `extract_amp_phase` vs. manual FFT, Sec. 2.6.1 |
| `THICKNESS_DENOM_ORDER` | `"clens_minus_cwater"` / `"cwater_minus_clens"` | sign convention in Eq. 2.11 |
| `ACCOUNT_FOR_LENS_THICKNESS` | `True` / `False` | whether target/skull placement is shifted by the lens standoff, Sec. 2.6.3 |
| `medium_mode` (driver, bottom of notebook) | `"free_field"` / `"skull"` | which of the two simulation configurations in Sec. 2.4.2 runs |

All of them default to the values used for the results reported in this thesis (read from
environment variables so they can also be overridden from the HPC job script), so the same
notebook reproduces every reported configuration by changing only one variable at a time.

In [26]:

import os
import numpy as np
import matplotlib.pyplot as plt
import sys
# sys.path.insert(0, os.path.expanduser("~/python_packages"))  # DISABLED: shadows kwave_env correct scipy/numpy
import nrrd
from scipy.ndimage import zoom, binary_erosion

from kwave.kgrid    import kWaveGrid
from kwave.kmedium  import kWaveMedium
from kwave.ksource  import kSource
from kwave.ksensor  import kSensor
from kwave.options.simulation_options           import SimulationOptions
from kwave.options.simulation_execution_options import SimulationExecutionOptions
from kwave.kspaceFirstOrder3D import kspaceFirstOrder3DC
from kwave.utils.signals import tone_burst
import plotly.graph_objects as go
import pandas as pd

## 1. Configuration & run parameters (Section 2.4.1)

Sets the transducer geometry, drive frequency/pressure/waveform, lens-standoff and
phase-extraction switches described above, builds the `RUN_TAG` used to name every
output file, and writes `run_parameters.txt` for provenance.

In [27]:

# ------------------------------------------------------------------
# OUTPUT_ROOT -- ONE shared parent folder for both media's output (per user request), replacing
# the two separate roots (phase_extraction_check / phase_extraction_skull) the source notebooks
# used. Override with PHOCUS_OUTPUT_ROOT.
# ------------------------------------------------------------------
OUTPUT_ROOT = os.environ.get(
    "PHOCUS_OUTPUT_ROOT",
    r'c:\Users\lucia\OneDrive - Imperial College London\msc project\Kwave simulations\phase_extraction_combined'
)

CT_DIR = os.environ.get(
    "PHOCUS_CT_DIR",
    r'c:\Users\lucia\OneDrive - Imperial College London\msc project\Kwave simulations'
)
CT_PATH = os.path.join(CT_DIR, 'DMBA_N20_230328-4-1_CT-cropped_M4D.nhdr')

# ------------------------------------------------------------------
# Transducer (Murphy 2022 -- Thorlabs PA44LE)
# ------------------------------------------------------------------
F0       = float(os.environ.get("PHOCUS_F0",       "625e3"))   # Hz
OD       = 8.3e-3       # m
ID       = 3.0e-3       # m
P0       = float(os.environ.get("PHOCUS_P0",       "1e6"))    # Pa
N_CYCLES = int(os.environ.get("PHOCUS_N_CYCLES",   "20"))
# Source notebooks defaulted to different values ("toneburst"/1 free-field, "morlet"/2 skull) --
# for a fair free-field-vs-skull comparison in one run, one consistent physical configuration is
# used for both branches: bilateral targets, tone-burst drive. Both still overridable.
SOURCE_WAVEFORM   = os.environ.get("PHOCUS_SOURCE_WAVEFORM", "morlet")   # "toneburst" | "morlet" | "sinusoid"
N_VIRTUAL_TARGETS = int(os.environ.get("PHOCUS_N_VIRTUAL_TARGETS", "2"))    # always 2 (bilateral) here
SINGLE_TARGET_SIDE = os.environ.get("PHOCUS_SINGLE_TARGET_SIDE", "left")   # unused when N_VIRTUAL_TARGETS==2

# Water (Murphy et al. 2022 / COMSOL values)
C_WATER   = 1485.0
RHO_WATER =  997.0

# ------------------------------------------------------------------
# Lens-standoff compensation for the virtual-source design targets (Sec. 2.6.3)
# ------------------------------------------------------------------
os.environ["PHOCUS_ACCOUNT_FOR_LENS_THICKNESS"] = "true"
_env_lens_thk = os.environ.get("PHOCUS_ACCOUNT_FOR_LENS_THICKNESS")
ACCOUNT_FOR_LENS_THICKNESS = (_env_lens_thk.strip().lower() in ("1", "true", "yes")
                               if _env_lens_thk is not None else True)

_env_lens_thickness_mm = os.environ.get("PHOCUS_LENS_THICKNESS_MM")
_f0_khz = F0 / 1e3
if _env_lens_thickness_mm is not None:
    LENS_THICKNESS_MM = float(_env_lens_thickness_mm)
elif abs(_f0_khz - 625.0) < 1e-6:
    LENS_THICKNESS_MM = 5.55
elif abs(_f0_khz - 1000.0) < 1e-6:
    LENS_THICKNESS_MM = 3.45
else:
    LENS_THICKNESS_MM = 5.55
    print(f"WARNING: no known lens thickness for F0={_f0_khz:.1f} kHz -- "
          f"falling back to LENS_THICKNESS_MM={LENS_THICKNESS_MM} mm. "
          f"Set PHOCUS_LENS_THICKNESS_MM to override.")

# Source notebooks defaulted to different phase-extraction methods too ("kwave" free-field,
# "manual" skull) -- again pinned to one consistent choice for a fair comparison.
PHASE_EXTRACTION_METHOD = os.environ.get("PHOCUS_PHASE_EXTRACTION_METHOD", "kwave")   # "manual" or "kwave"

THICKNESS_DENOM_ORDER = os.environ.get("PHOCUS_THICKNESS_DENOM_ORDER", "clens_minus_cwater")
PHASE_SIGN_INVERTED = (PHASE_EXTRACTION_METHOD == "manual")

# ------------------------------------------------------------------
# RUN_TAG / OUTPUT_DIR -- ONE shared folder for both media (per-medium filenames get a
# _freefield / _skull suffix everywhere below instead).
# ------------------------------------------------------------------
_lens_tag = (
    f"lensThk{LENS_THICKNESS_MM:.2f}mm".replace('.', 'p')
    if ACCOUNT_FOR_LENS_THICKNESS else "noLensThk"
)
RUN_TAG = (
    f"F0-{F0/1e3:.0f}kHz"
    f"_P0-{P0/1e6:.2f}MPa"
    f"_Ncyc-{N_CYCLES}"
    f"_{_lens_tag}"
    f"_phase-{PHASE_EXTRACTION_METHOD}"
    f"_phaseInv-{'Y' if PHASE_SIGN_INVERTED else 'N'}"
    f"_denom-{'CLmCW' if THICKNESS_DENOM_ORDER == 'clens_minus_cwater' else 'CWmCL'}"
    f"_wave-{SOURCE_WAVEFORM}"
    f"_ntgt-{N_VIRTUAL_TARGETS}"
    f"{('-' + SINGLE_TARGET_SIDE) if N_VIRTUAL_TARGETS == 1 else ''}"
    f"_bothMedia"
).replace('.', 'p')

def run_tag_pretty(medium_label):
    return (
        f"{F0/1e3:.0f} kHz, P$_0$={P0/1e6:.2f} MPa, {N_CYCLES} cyc, "
        + (f"lens +{LENS_THICKNESS_MM:.2f} mm" if ACCOUNT_FOR_LENS_THICKNESS else "no lens standoff")
        + f", phase={PHASE_EXTRACTION_METHOD} (sign {'inverted' if PHASE_SIGN_INVERTED else 'as-extracted'})"
        + f", denom={'c_lens-c_water' if THICKNESS_DENOM_ORDER == 'clens_minus_cwater' else 'c_water-c_lens'}"
        + f", wave={SOURCE_WAVEFORM}"
        + f", medium={medium_label}"
    )

print(f"RUN_TAG = {RUN_TAG}")

OUTPUT_DIR = os.path.join(OUTPUT_ROOT, RUN_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OUTPUT_DIR = {OUTPUT_DIR}")

with open(os.path.join(OUTPUT_DIR, "run_parameters.txt"), "w") as _f:
    _f.write("PhoCUS run parameters (BOTH media, one run)\n")
    _f.write("=" * 50 + "\n")
    _f.write(f"RUN_TAG                     : {RUN_TAG}\n")
    _f.write(f"F0 (Hz)                     : {F0}\n")
    _f.write(f"N_CYCLES                    : {N_CYCLES}\n")
    _f.write(f"P0 (Pa)                     : {P0}\n")
    _f.write(f"OD / ID (m)                 : {OD} / {ID}\n")
    _f.write(f"C_WATER / RHO_WATER         : {C_WATER} / {RHO_WATER}\n")
    _f.write(f"ACCOUNT_FOR_LENS_THICKNESS  : {ACCOUNT_FOR_LENS_THICKNESS}\n")
    _f.write(f"LENS_THICKNESS_MM           : {LENS_THICKNESS_MM}\n")
    _f.write(f"PHASE_EXTRACTION_METHOD     : {PHASE_EXTRACTION_METHOD}\n")
    _f.write(f"THICKNESS_DENOM_ORDER       : {THICKNESS_DENOM_ORDER}\n")
    _f.write(f"SOURCE_WAVEFORM             : {SOURCE_WAVEFORM}\n")
    _f.write(f"N_VIRTUAL_TARGETS           : {N_VIRTUAL_TARGETS}\n")

RUN_TAG = F0-625kHz_P0-1p00MPa_Ncyc-20_lensThk5p55mm_phase-kwave_phaseInv-N_denom-CLmCW_wave-morlet_ntgt-2_bothMedia
OUTPUT_DIR = c:\Users\lucia\OneDrive - Imperial College London\msc project\Kwave simulations\phase_extraction_combined\F0-625kHz_P0-1p00MPa_Ncyc-20_lensThk5p55mm_phase-kwave_phaseInv-N_denom-CLmCW_wave-morlet_ntgt-2_bothMedia


## 1.1 Excitation waveform options (Section 2.6.1)

`generate_source_signal` is the single source of truth for every drive waveform: the
toneburst, Morlet-windowed toneburst, and sinusoid options described in Sec. 2.6.1
(the sinusoid excludes itself for phase extraction in the thesis text, but is kept here
as a selectable option via `SOURCE_WAVEFORM`).

In [28]:

# ------------------------------------------------------------------
# generate_source_signal -- single source of truth for every drive waveform.
# ------------------------------------------------------------------
def generate_source_signal(waveform, fs, f0, n_cycles, p0=1.0, t_array=None):
    waveform = waveform.lower()
    if waveform == 'toneburst':
        sig = p0 * tone_burst(fs, f0, n_cycles)
    elif waveform == 'morlet':
        morlet_cycles = n_cycles * 2
        sigma = morlet_cycles / (2 * np.pi * f0)
        shift = (morlet_cycles / 2) / f0
        n_samples = int(round(morlet_cycles / f0 * fs)) + 1
        t = np.arange(n_samples) / fs
        sig = p0 * np.real(-np.exp(2j * np.pi * f0 * (t - shift)) *
                            np.exp(-(t - shift) ** 2 / (2 * sigma ** 2)))
    elif waveform == 'sinusoid':
        if t_array is None:
            raise ValueError("SOURCE_WAVEFORM='sinusoid' needs t_array=kgrid.t_array of the "
                              "destination grid.")
        t = np.asarray(t_array).ravel()
        ramp_cycles = min(3, n_cycles)
        ramp_n = max(1, int(round(ramp_cycles / f0 * fs)))
        ramp = np.ones_like(t)
        ramp[:ramp_n] = 0.5 * (1 - np.cos(np.pi * np.arange(ramp_n) / ramp_n))
        sig = p0 * np.sin(2 * np.pi * f0 * t) * ramp
    else:
        raise ValueError(f"Unknown SOURCE_WAVEFORM '{waveform}' -- use 'toneburst', 'morlet', or 'sinusoid'.")
    return np.atleast_2d(sig).astype(np.float32)

print(f"SOURCE_WAVEFORM = '{SOURCE_WAVEFORM}'")

SOURCE_WAVEFORM = 'morlet'


## 2. Skull acquisition, segmentation and acoustic properties assignment (Section 2.1)
### Load the micro-CT skull volume (DMBA)

In [29]:

def load_ct_volume(path):
    def build_ct_affine(header):
        directions = np.array(header['space directions'], dtype=float)
        origin     = np.array(header['space origin'],      dtype=float)
        def world_to_ijk(world):
            world = np.asarray(world, dtype=float)
            return np.linalg.solve(directions.T, (world - origin))
        voxel_size_mm = np.abs(np.diag(directions)) if np.allclose(directions, np.diag(np.diag(directions))) \
                        else np.linalg.norm(directions, axis=1)
        return world_to_ijk, voxel_size_mm
    data, header = nrrd.read(path)
    world_to_ijk, voxel_size_mm = build_ct_affine(header)
    return data, header, world_to_ijk, voxel_size_mm

ct_raw, ct_header, ct_world_to_ijk, ct_voxel_mm = load_ct_volume(CT_PATH)
print(f"CT array shape (i, j, k)  = {ct_raw.shape}")
print(f"Voxel size (mm)           = {ct_voxel_mm}")

CT array shape (i, j, k)  = (501, 696, 374)
Voxel size (mm)           = [0.02501 0.02501 0.02501]


### 2.1 CT to acoustic properties (Section 2.1, Eq. 2.1-2.2, Table 2.1)

The CT volume holds native grayscale intensities, not calibrated Hounsfield units, so there is
no HU-based property mapping available -- only the percentile calibration of Eq. 2.1 is used.
`ct_to_rescaled_intensity` rescales the CT volume this way (returning `rescaled_intensity`,
*not* a physical porosity map), which is then thresholded into a binary skull mask (Eq. 2.2)
before assigning the homogeneous bone/water density, sound speed and attenuation of Table 2.1.

Density & bulk longitudinal speed: Estrada et al. 2016, Table 2 (avg of 2 ex-vivo adult mouse
skulls) -> rho = 1951 kg/m^3, c = 2334 m/s. Absorption: Jimenez et al. 2022 states, verbatim,
"28.3 dB/cm at 1.68 MHz with a frequency-power law exponent of gamma = 1" -- LINEAR, so
BONE_ALPHA0 = 28.3 / 1.68 is the calibrated coefficient under gamma=1. ALPHA_POWER is 1.01
rather than exactly 1 (k-Wave's power-law absorption model is singular at y=1). k-Wave only
accepts one global alpha_power for the whole medium, so this same exponent is reused for the
lens resin's LENS_ALPHA0 once combined with the skull into medium_lens (Sec. 10.1).

In [30]:
ALPHA_POWER  = 1.01
BONE_DENSITY = 0.5 * (1969.0 + 1933.0)   # kg/m^3  (= 1951)
BONE_SPEED   = 0.5 * (2242.0 + 2425.0)   # m/s     (= 2334)
BONE_ALPHA0  = 28.3 / 1.68               # dB/(MHz . cm), gamma = 1 (Jimenez et al. 2022)

WATER_ALPHA0 = 0.0   # negligible over these path lengths

INTENSITY_THRESHOLD = 0.5     # rescaled_intensity below this -> "skull" in the binary mask

print(f"BONE_DENSITY = {BONE_DENSITY:.0f} kg/m^3   (Estrada 2016, Table 2)")
print(f"BONE_SPEED   = {BONE_SPEED:.0f} m/s        (Estrada 2016, Table 2)")
print(f"BONE_ALPHA0  = {BONE_ALPHA0:.2f} dB/(MHz.cm)   (Jimenez et al. 2022, gamma=1)")
print(f"  -> at F0={F0/1e6:.3f} MHz: alpha_bone = {BONE_ALPHA0*(F0/1e6)**ALPHA_POWER:.2f} dB/cm")


def build_acoustic_maps(rescaled_intensity):
    skull_mask = rescaled_intensity < INTENSITY_THRESHOLD
    density = np.where(skull_mask, BONE_DENSITY, RHO_WATER).astype(np.float32)
    speed   = np.where(skull_mask, BONE_SPEED,   C_WATER).astype(np.float32)
    alpha   = np.where(skull_mask, BONE_ALPHA0,  WATER_ALPHA0).astype(np.float32)
    return density, speed, alpha, skull_mask


CT_PCTL_LOW, CT_PCTL_HIGH = 1.0, 99.5

def ct_to_rescaled_intensity(ct, pctl_low=CT_PCTL_LOW, pctl_high=CT_PCTL_HIGH):
    # Percentile-normalised CT intensity (Eq. 2.1), inverted so low rescaled_intensity = bone,
    # matching the sense of the INTENSITY_THRESHOLD comparisons below.
    ct = ct.astype(np.float32)
    lo, hi = np.percentile(ct, [pctl_low, pctl_high])
    rescaled_intensity = 1.0 - (ct - lo) / max(hi - lo, 1e-6)
    return np.clip(rescaled_intensity, 0.0, 1.0).astype(np.float32)

rescaled_intensity_full = ct_to_rescaled_intensity(ct_raw)
density_full, speed_full, alpha_full, skull_mask_full = build_acoustic_maps(rescaled_intensity_full)
print(f"Skull-mask coverage : {skull_mask_full.mean()*100:.1f}%")

BONE_DENSITY = 1951 kg/m^3   (Estrada 2016, Table 2)
BONE_SPEED   = 2334 m/s        (Estrada 2016, Table 2)
BONE_ALPHA0  = 16.85 dB/(MHz.cm)   (Jimenez et al. 2022, gamma=1)
  -> at F0=0.625 MHz: alpha_bone = 10.48 dB/cm
Skull-mask coverage : 4.0%


### 2.2 Skull resampling onto the simulation grid (Sections 2.1, 2.3 from Methods; dx = 75 um)

In [31]:

DX_SIM = 75e-6    # m -- simulation grid spacing

zoom_factor = ct_voxel_mm * 1e-3 / DX_SIM
rescaled_intensity_sim = zoom(rescaled_intensity_full, zoom_factor, order=1)
rescaled_intensity_sim = rescaled_intensity_sim[:, :, ::-1]
print(f"Resampled skull sub-volume: {ct_raw.shape} @ {ct_voxel_mm[0]*1e3:.1f} um  ->  "
      f"{rescaled_intensity_sim.shape} @ {DX_SIM*1e6:.0f} um")

bone_z_profile_native = np.any(rescaled_intensity_full[:, :, ::-1] < INTENSITY_THRESHOLD, axis=(0, 1))
first_bone_z_mm = int(np.argmax(bone_z_profile_native)) * ct_voxel_mm[2]
first_bone_z = int(round(first_bone_z_mm / (DX_SIM * 1e3)))
print(f"Empty margin before real bone starts: {first_bone_z_mm:.4f} mm (native)  ->  "
      f"{first_bone_z} vox @ {DX_SIM*1e6:.0f} um")
rescaled_intensity_sim = rescaled_intensity_sim[:, :, first_bone_z:]

# Fixed physical domain size (mm), tuned at DX_SIM=75um so the resampled skull fits comfortably.
DOMAIN_X_MM, DOMAIN_Y_MM, DOMAIN_Z_MM = 13.4, 18, 15.5
Nx = int(round(DOMAIN_X_MM * 1e-3 / DX_SIM))
Ny = int(round(DOMAIN_Y_MM * 1e-3 / DX_SIM))
Nz = int(round(DOMAIN_Z_MM * 1e-3 / DX_SIM))

lam_water = C_WATER / F0
print(f"Grid   = {Nx} x {Ny} x {Nz}  = {Nx*Ny*Nz/1e6:.2f} M voxels")
print(f"Domain = {Nx*DX_SIM*1e3:.1f} x {Ny*DX_SIM*1e3:.1f} x {Nz*DX_SIM*1e3:.1f} mm")
print(f"PPW (water) = {lam_water/DX_SIM:.1f}")

Resampled skull sub-volume: (501, 696, 374) @ 25.0 um  ->  (167, 232, 125) @ 75 um
Empty margin before real bone starts: 1.1255 mm (native)  ->  15 vox @ 75 um
Grid   = 179 x 240 x 207  = 8.89 M voxels
Domain = 13.4 x 18.0 x 15.5 mm
PPW (water) = 31.7


### 2.3 Embed the skull in the simulation domain & define the k-Wave media (Section 2.1)

Shared between both media: the free-field branch (`medium_free`, pure water) and the
transcranial branch (`medium_skull`) are both built here; only one is actually driven
through `kspaceFirstOrder3DC` per branch, selected by `medium_mode` in the driver cell
at the end of this notebook.

In [32]:

if ACCOUNT_FOR_LENS_THICKNESS:
    Z_SKULL_START_MM = LENS_THICKNESS_MM
else:
    Z_SKULL_START_MM = 0

LATERAL_OFFSET_VOX = (0, 0)

def paste_skull_into_domain(intensity_sub, Nx, Ny, Nz, z_start_mm, dx_sim, lateral_offset_vox=(0, 0)):
    domain = np.ones((Nx, Ny, Nz), dtype=np.float32)   # rescaled_intensity = 1 everywhere = water
    sx, sy, sz = intensity_sub.shape
    z0 = int(round(z_start_mm * 1e-3 / dx_sim))
    x0 = (Nx - sx) // 2 + lateral_offset_vox[0]
    y0 = (Ny - sy) // 2 + lateral_offset_vox[1]
    x0c, y0c, z0c = max(x0, 0), max(y0, 0), max(z0, 0)
    x1c, y1c, z1c = min(x0 + sx, Nx), min(y0 + sy, Ny), min(z0 + sz, Nz)
    if x1c <= x0c or y1c <= y0c or z1c <= z0c:
        raise ValueError("Skull sub-volume placement falls entirely outside the domain -- "
                          "check Z_SKULL_START_MM / Nx,Ny,Nz / DX_SIM.")
    sub_x0, sub_y0, sub_z0 = x0c - x0, y0c - y0, z0c - z0
    domain[x0c:x1c, y0c:y1c, z0c:z1c] = intensity_sub[
        sub_x0:sub_x0 + (x1c - x0c), sub_y0:sub_y0 + (y1c - y0c), sub_z0:sub_z0 + (z1c - z0c)]
    placement = dict(x0=x0c, x1=x1c, y0=y0c, y1=y1c, z0=z0c, z1=z1c)
    return domain, placement

rescaled_intensity_domain, skull_placement = paste_skull_into_domain(
    rescaled_intensity_sim, Nx, Ny, Nz, Z_SKULL_START_MM, DX_SIM, LATERAL_OFFSET_VOX)
density_domain, speed_domain, alpha_domain, skull_mask_domain = build_acoustic_maps(rescaled_intensity_domain)
print("Skull placed at voxel range:", skull_placement)

# Both media's kWaveMedium objects are cheap to build unconditionally -- only ONE of them is
# actually driven through kspaceFirstOrder3DC per branch below.
medium_skull = kWaveMedium(sound_speed=speed_domain, density=density_domain,
                            alpha_coeff=alpha_domain, alpha_power=ALPHA_POWER)
medium_free  = kWaveMedium(sound_speed=C_WATER, density=RHO_WATER)

C_MAX_DOMAIN = float(max(C_WATER, speed_domain.max()))
T_END = 40e-6
print(f"[Skull] C_MAX_DOMAIN = {C_MAX_DOMAIN:.0f} m/s   |   [Free] C_MAX_DOMAIN = {C_WATER:.0f} m/s")

Skull placed at voxel range: {'x0': 6, 'x1': 173, 'y0': 4, 'y1': 236, 'z0': 74, 'z1': 184}
[Skull] C_MAX_DOMAIN = 2334 m/s   |   [Free] C_MAX_DOMAIN = 1485 m/s


## 3. Hippocampal target localisation (Section 2.2)

Bregma and the bilateral CA1 targets (`fus_plan_left.csv` / `fus_plan_right.csv`, from the
atlas-based region-labelling tool) are transformed into simulation-grid coordinates using
the same resampling/axis-flip/cropping/placement pipeline applied to the CT volume above.
Shared between both media (identical target geometry for free-field and transcranial runs).

In [33]:

i0, j0, k0 = 242, 395, 312   # bregma, manually located in 3D Slicer, raw CT voxel index

xc, yc, z_src = Nx // 2, Ny // 2, 0
Z_TOTAL_MM = ct_raw.shape[2] * ct_voxel_mm[2]

i_center_native = ct_raw.shape[0] / 2
j_center_native = ct_raw.shape[1] / 2

def raw_ijk_to_grid_absolute(i_raw, j_raw, k_raw):
    '''Continuous raw-CT-voxel index -> sim-grid dict, referenced to the native CT volume's
    own geometric centre, single-rounding at the end onto the current DX_SIM.'''
    x_mm = (i_raw - i_center_native) * ct_voxel_mm[0]
    y_mm = (j_raw - j_center_native) * ct_voxel_mm[1]
    z_mm = (Z_TOTAL_MM - k_raw * ct_voxel_mm[2]) - first_bone_z_mm
    return dict(
        x_idx=xc + int(round(x_mm / (DX_SIM * 1e3))),
        y_idx=yc + int(round(y_mm / (DX_SIM * 1e3))),
        z_idx=z_src + int(round(z_mm / (DX_SIM * 1e3))),
        x_mm=x_mm, y_mm=y_mm, z_mm=z_mm,
    )

_bregma = raw_ijk_to_grid_absolute(i0, j0, k0)
bregma_x_idx, bregma_y_idx, bregma_z_idx = _bregma['x_idx'], _bregma['y_idx'], _bregma['z_idx']
bregma_x_mm, bregma_y_mm, bregma_z_mm = _bregma['x_mm'], _bregma['y_mm'], _bregma['z_mm']
print(f"Bregma located at grid index: ({bregma_x_idx}, {bregma_y_idx}, {bregma_z_idx})")
assert 0 <= bregma_x_idx < Nx and 0 <= bregma_y_idx < Ny and 0 <= bregma_z_idx < Nz, \
    "Bregma projects outside the domain -- check Nx, Ny, Nz, or DX_SIM (too coarse)."

x_mm3 = (np.arange(Nx) - xc)    * DX_SIM * 1e3
y_mm3 = (np.arange(Ny) - yc)    * DX_SIM * 1e3
z_mm3 = (np.arange(Nz) - z_src) * DX_SIM * 1e3

FUS_PLAN_LEFT_CSV = os.environ.get(
    "PHOCUS_FUS_PLAN_LEFT_CSV",
    r"C:\Users\lucia\OneDrive - Imperial College London\msc project\CT DATA\DMBA\fus_plan_left.csv"
)
FUS_PLAN_RIGHT_CSV = os.environ.get(
    "PHOCUS_FUS_PLAN_RIGHT_CSV",
    r"C:\Users\lucia\OneDrive - Imperial College London\msc project\CT DATA\DMBA\fus_plan_right.csv"
)

def target_from_fus_plan(csv_path, label):
    row = pd.read_csv(csv_path).iloc[0]
    i_raw, j_raw, k_raw = ct_world_to_ijk(
        [row['world_x_lps_mm'], row['world_y_lps_mm'], row['world_z_lps_mm']])
    target = raw_ijk_to_grid_absolute(i_raw, j_raw, k_raw)
    assert 0 <= target['x_idx'] < Nx and 0 <= target['y_idx'] < Ny, f"{label} target index outside domain"
    print(f"{label} target (from {os.path.basename(csv_path)}): "
          f"grid index ({target['x_idx']}, {target['y_idx']}, {target['z_idx']})  "
          f"[{target['x_mm']:+.2f}, {target['y_mm']:+.2f}, {target['z_mm']:.2f}] mm")
    return target

target_left  = target_from_fus_plan(FUS_PLAN_LEFT_CSV,  "Left")
target_right = target_from_fus_plan(FUS_PLAN_RIGHT_CSV, "Right")

Bregma located at grid index: (86, 136, 6)
Left target (from fus_plan_left.csv): grid index (52, 100, 42)  [-2.75, -1.52, 3.13] mm
Right target (from fus_plan_right.csv): grid index (121, 99, 42)  [+2.39, -1.54, 3.18] mm


### Reference only: manual bregma-based target definition (Appendix B from Methods)

Not executed -- the atlas-based targets above are what is actually used. Before that tool
was available, targets were located manually from bregma plus literature stereotactic
offsets; that method is retained as a reproducible fallback (`phocus_transducer_validation.ipynb`,
Section 5.1) and reproduced here, commented out, purely for reference.

In [34]:

# AP_MM, ML_MM, DEPTH_MM = -1.8, 1.1, 1.9   # stereotactic offsets from bregma (mm)
#
# def skull_surface_z(x_idx, y_idx, mask, z_mm):
#     '''Z (mm) of the first skull voxel along +Z at grid column (x_idx, y_idx).'''
#     hits = np.flatnonzero(mask[x_idx, y_idx, :])
#     return z_mm[hits[0]] if len(hits) else np.nan
#
# def stereotactic_to_grid(ap_mm, ml_mm_signed, depth_mm, x_mm, y_mm, z_mm, skull_mask):
#     '''Offsets are measured from the real bregma position computed above
#     (bregma_x_mm, bregma_y_mm), not the domain centre. depth_mm is measured from THIS
#     column's own local skull surface, not bregma's.'''
#     x_idx = int(np.argmin(np.abs(x_mm - (bregma_x_mm + ml_mm_signed))))
#     y_idx = int(np.argmin(np.abs(y_mm - (bregma_y_mm + ap_mm))))
#     surf_z_mm = skull_surface_z(x_idx, y_idx, skull_mask, z_mm)
#     z_idx = int(np.argmin(np.abs(z_mm - (surf_z_mm + depth_mm))))
#     return dict(x_idx=x_idx, y_idx=y_idx, z_idx=z_idx,
#                 x_mm=x_mm[x_idx], y_mm=y_mm[y_idx], z_mm=z_mm[z_idx], surf_z_mm=surf_z_mm)
#
# target_right = stereotactic_to_grid(AP_MM, +ML_MM, DEPTH_MM, x_mm3, y_mm3, z_mm3, skull_mask_domain)
# target_left  = stereotactic_to_grid(AP_MM, -ML_MM, DEPTH_MM, x_mm3, y_mm3, z_mm3, skull_mask_domain)

## 4. Transducer geometry (Section 2.4.1 )

The PhoCUS annular source mask (8.3 mm outer / 3.0 mm inner diameter, Table 2.2), i.e. the
physical holographic surface where the lens sits.

In [35]:

# Ring geometry (annulus mask) -- the physical holographic surface where the lens sits.
TARGET_Y_IDX = target_right['y_idx']   # == target_left['y_idx'], same AP for both
_x = (np.arange(Nx) - xc) * DX_SIM
_y = (np.arange(Ny) - TARGET_Y_IDX) * DX_SIM
_X, _Y = np.meshgrid(_x, _y, indexing='ij')
_R = np.sqrt(_X**2 + _Y**2)
ring2d = (_R >= ID / 2) & (_R <= OD / 2)
src_mask = np.zeros((Nx, Ny, Nz), dtype=bool)
src_mask[:, :, z_src] = ring2d
n_src = int(ring2d.sum())

## 5. Simulation configurations: shared execution options (Section 2.4.2)

`SimulationOptions`/`SimulationExecutionOptions` shared by both the phase-extraction and
forward-propagation configurations of Sec. 2.4.2 (source/sensor roles differ per branch,
defined inside `run_lens_pipeline` below).

In [36]:

sim_opts_common = dict(pml_inside=False, data_cast='single', save_to_disk=True)
exec_opts = SimulationExecutionOptions(is_gpu_simulation=False, delete_data=False, verbose_level=1)

## 6. Virtual-source time reversal: lens-standoff design targets (Section 2.6.1)

Mode-independent (shared by both media): the lens-standoff shift depends only on
`LENS_THICKNESS_MM`/`ACCOUNT_FOR_LENS_THICKNESS` (Sec. 2.6.3), not on which medium the
phase-extraction simulation propagates through.

In [37]:

# ------------------------------------------------------------------
# Virtual-source DESIGN targets (Sec. 9.1) -- mode-independent: the lens-standoff shift depends
# only on LENS_THICKNESS_MM/ACCOUNT_FOR_LENS_THICKNESS, not on which medium the phase-extraction
# sim propagates through, so this is computed once and shared by both branches.
# ------------------------------------------------------------------
def _shift_target_z(target, z_shift_mm, z_mm):
    new_z_mm = target['z_mm'] + z_shift_mm
    z_idx = int(np.argmin(np.abs(z_mm - new_z_mm)))
    return dict(target, z_idx=z_idx, z_mm=z_mm[z_idx])

if ACCOUNT_FOR_LENS_THICKNESS:
    target_left_design  = _shift_target_z(target_left,  LENS_THICKNESS_MM, z_mm3)
    target_right_design = _shift_target_z(target_right, LENS_THICKNESS_MM, z_mm3)
    bregma_z_mm_design = bregma_z_mm + Z_SKULL_START_MM
else:
    target_left_design  = target_left
    target_right_design = target_right
    bregma_z_mm_design = bregma_z_mm

virtual_mask = np.zeros((Nx, Ny, Nz), dtype=bool)
virtual_mask[target_left_design['x_idx'],  target_left_design['y_idx'],  target_left_design['z_idx']]  = True
virtual_mask[target_right_design['x_idx'], target_right_design['y_idx'], target_right_design['z_idx']] = True
n_virtual = int(virtual_mask.sum())
assert n_virtual == 2, f"expected 2 virtual sources, got {n_virtual}"
print(f"Virtual monopole sources (placed {LENS_THICKNESS_MM:.2f} mm deeper, to account for the lens):")
print(f"  L: x={target_left_design['x_mm']:+.2f} mm  z={target_left_design['z_mm']:.2f} mm")
print(f"  R: x={target_right_design['x_mm']:+.2f} mm  z={target_right_design['z_mm']:.2f} mm")

Virtual monopole sources (placed 5.55 mm deeper, to account for the lens):
  L: x=-2.75 mm  z=8.70 mm
  R: x=+2.39 mm  z=8.70 mm


## 7. Shared analysis & visualisation utilities (supports Results Ch. 3)

Helper functions used throughout the Results pipeline below: FWHM (Sec. 2.5 validation
metric, applied here to the lens-verification field in Ch. 3), slice plotting, and
per-focus peak-pressure/FWHM metrics (Sec. 3.3 Acoustic performance of the bilateral lens).

In [38]:

x_mm = (np.arange(Nx) - xc)           * DX_SIM * 1e3
y_mm = (np.arange(Ny) - TARGET_Y_IDX) * DX_SIM * 1e3
z_mm = (np.arange(Nz) - z_src)        * DX_SIM * 1e3

def fwhm(prof, axis_mm):
    pk = prof.max()
    if pk <= 0: return np.nan, np.nan, np.nan
    above = np.where(prof >= pk / 2)[0]
    if len(above) < 2: return np.nan, np.nan, np.nan
    return axis_mm[above[-1]] - axis_mm[above[0]], axis_mm[above[0]], axis_mm[above[-1]]


def show_sim_slice(volume, axis, index=None, coord_mm=None, ax=None, cmap='hot',
                    title=None, extent_mm=None, overlay_mask=None, overlay_color='cyan', **imkw):
    axis_map = {'x': 0, 'y': 1, 'z': 2}
    ax_idx = axis_map[axis]
    axes_mm = {'x': x_mm, 'y': y_mm, 'z': z_mm}
    if coord_mm is not None:
        index = int(np.argmin(np.abs(axes_mm[axis] - coord_mm)))
    if index is None:
        index = volume.shape[ax_idx] // 2
    sl = [slice(None)] * 3
    sl[ax_idx] = index
    img = volume[tuple(sl)]
    other_axes = [a for a in ('x', 'y', 'z') if a != axis]
    if extent_mm is None:
        extent_mm = [axes_mm[other_axes[0]][0], axes_mm[other_axes[0]][-1],
                     axes_mm[other_axes[1]][-1], axes_mm[other_axes[1]][0]]
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 6.5))
    im = ax.imshow(img.T, extent=extent_mm, aspect='equal', origin='upper', cmap=cmap, **imkw)
    if overlay_mask is not None:
        mask_slice = overlay_mask[tuple(sl)].astype(float)
        if mask_slice.max() > 0:
            ax.contour(mask_slice.T, levels=[0.5], extent=extent_mm, origin='upper',
                       colors=overlay_color, linewidths=1.2)
    ax.set_xlabel(f'{other_axes[0]} (mm)'); ax.set_ylabel(f'{other_axes[1]} (mm)')
    ax.set_title(title or f'{axis} = {axes_mm[axis][index]:.2f} mm  (index {index})')
    plt.colorbar(im, ax=ax, shrink=0.8)
    return ax, index


def per_focus_metrics(p_field, x_idx, z_idx, y_idx, label):
    axial   = p_field[x_idx, y_idx, :]
    lateral = p_field[:, y_idx, z_idx]
    fwhm_ax,  *_ = fwhm(axial,  z_mm)
    fwhm_lat, *_ = fwhm(lateral, x_mm)
    print(f"--- {label} ---")
    print(f"  Target (x, z) : ({x_mm[x_idx]:+.2f}, {z_mm[z_idx]:.2f}) mm")
    print(f"  Peak negative pressure (PNP) : {p_field[x_idx, y_idx, z_idx]/1e3:.1f} kPa")
    print(f"  FWHM axial  (-6 dB): {fwhm_ax:.2f} mm")
    print(f"  FWHM lateral(-6 dB): {fwhm_lat:.2f} mm")
    return dict(x_idx=x_idx, z_idx=z_idx, x_mm=x_mm[x_idx], z_mm=z_mm[z_idx],
                peak_kPa=p_field[x_idx, y_idx, z_idx] / 1e3,
                fwhm_ax_mm=fwhm_ax, fwhm_lat_mm=fwhm_lat)

## 8. Main pipeline (Sections 2.4.2, 2.6, 2.7; Ch. 3 from Results)

`run_lens_pipeline(medium_mode)` runs the full per-medium pipeline -- virtual-source phase
extraction, lens design, lens build, verification simulation, metrics and figures -- for one
medium. It is broken into one function per thesis subsection below -- each function is its
own cell, titled to match -- then chained together by the short orchestrator function at
the end via a shared `ctx` dict that carries intermediate results from one step to the next.

In [39]:

# ================================================================================
# END SHARED SETUP -- everything below runs once per medium via run_lens_pipeline()
# ================================================================================

### 8.1 Virtual-source phase-extraction simulation (Sections 2.3, 2.6.1)

kgrid_tr's CFL is the one genuine physics difference here (heterogeneous high-contrast
skull needs a lower CFL for stability than homogeneous water).

In [40]:

def run_phase_extraction_sim(medium_mode, ctx):
    suffix, medium_label = ctx['suffix'], ctx['medium_label']

    kgrid_tr = kWaveGrid([Nx, Ny, Nz], [DX_SIM, DX_SIM, DX_SIM])
    if medium_mode == "skull":
        kgrid_tr.makeTime(C_MAX_DOMAIN, cfl=0.1, t_end=T_END)
    else:
        kgrid_tr.makeTime(C_WATER, cfl=0.3, t_end=T_END)

    sig_virtual = generate_source_signal(SOURCE_WAVEFORM, 1 / kgrid_tr.dt, F0, N_CYCLES, p0=1.0,
                                          t_array=kgrid_tr.t_array)
    source_virtual = kSource()
    source_virtual.p_mask = virtual_mask
    source_virtual.p      = np.tile(sig_virtual, (n_virtual, 1)).astype(np.float32)

    sensor_ring = kSensor()
    sensor_ring.mask   = src_mask
    sensor_ring.record = ['p']

    sim_opts_tr = SimulationOptions(
        input_filename  = os.path.join(OUTPUT_DIR, f'in_virtualsource_{suffix}.h5'),
        output_filename = os.path.join(OUTPUT_DIR, f'out_virtualsource_{suffix}.h5'),
        **sim_opts_common)

    medium_for_phase = medium_skull if medium_mode == "skull" else medium_free
    print(f"\n[{medium_label}] Running the virtual-source phase-extraction simulation...")
    out_virtual = kspaceFirstOrder3DC(
        kgrid=kgrid_tr, medium=medium_for_phase, source=source_virtual, sensor=sensor_ring,
        simulation_options=sim_opts_tr, execution_options=exec_opts)

    p_ring_raw = np.asarray(out_virtual['p'])
    if p_ring_raw.shape[0] == n_src:
        p_ring_t = p_ring_raw
    elif p_ring_raw.shape[1] == n_src:
        p_ring_t = p_ring_raw.T
    else:
        raise ValueError(f"Neither axis of out_virtual['p'] (shape {p_ring_raw.shape}) matches n_src={n_src}")
    print(f"Recorded {p_ring_t.shape[0]} ring-aperture sensor points, {p_ring_t.shape[1]} time samples each.")

    ctx['kgrid_tr'] = kgrid_tr
    ctx['p_ring_t'] = p_ring_t
    return ctx

### 8.2 Phase extraction (Section 2.6.1)

Pure math, identical for both media (only the recorded `p_ring_t` differs).

In [41]:

def extract_phase(medium_mode, ctx):
    kgrid_tr, p_ring_t = ctx['kgrid_tr'], ctx['p_ring_t']

    if PHASE_EXTRACTION_METHOD == "manual":
        Nt_tr, dt_tr = p_ring_t.shape[1], kgrid_tr.dt
        Nfft    = 2 ** int(np.ceil(np.log2(Nt_tr)))
        f_array = np.arange(Nfft) * (1 / (Nfft * dt_tr))
        f_idx   = int(np.argmin(np.abs(f_array - F0)))
        complex_freq = np.fft.fft(p_ring_t, n=Nfft, axis=1)[:, f_idx]
        phase_recorded = np.angle(complex_freq)
        phase_profile  = np.mod(-phase_recorded, 2 * np.pi)
        phase_profile  = -phase_profile
        print(f"Phase extracted via manual FFT (Nfft={Nfft}, bin={f_array[f_idx]/1e3:.2f} kHz).")
    elif PHASE_EXTRACTION_METHOD == "kwave":
        from kwave.utils.filters import extract_amp_phase
        Fs = 1 / kgrid_tr.dt
        _, phase_profile, f_used_kwave = extract_amp_phase(p_ring_t, Fs, F0, dim=1)
        phase_profile = -phase_profile
        print(f"Phase extracted via k-Wave extract_amp_phase() (bin = {f_used_kwave/1e3:.2f} kHz).")
    else:
        raise ValueError(f"Unknown PHASE_EXTRACTION_METHOD={PHASE_EXTRACTION_METHOD!r}")

    ctx['phase_profile'] = phase_profile
    return ctx

### 8.3 Lens material properties & phase-to-thickness conversion (Sections 2.6.2, 2.6.3; Table 2.3, Eq. 2.11-2.12)

Also produces the ring-aperture phase & thickness map figure.

In [42]:

def design_lens_and_thickness_map(medium_mode, ctx):
    suffix, medium_label, phase_profile = ctx['suffix'], ctx['medium_label'], ctx['phase_profile']

    # -- 2.6.2: lens material selection & acoustic properties --
    C_LENS      = 2599.0   # m/s -- Formlabs Form 3, Clear Resin (Kim et al. 2025, Table 2)
    RHO_LENS    = 1186.0   # kg/m^3
    LENS_ALPHA0 = 3.4      # dB/(MHz.cm), gamma = 1 (Kim et al. 2025, 500 kHz)
    H0_LENS     = 0.25e-3  # m -- minimum base thickness

    # -- 2.6.3: phase-to-thickness conversion --
    c_lens, c_water, f0 = C_LENS, C_WATER, F0
    _denom_terms = {"clens_minus_cwater": c_lens - c_water, "cwater_minus_clens": c_water - c_lens}
    thickness = phase_profile * c_lens * c_water / (2 * np.pi * f0 * _denom_terms[THICKNESS_DENOM_ORDER])
    thickness = thickness - np.min(thickness)

    ring_lin_idx = np.flatnonzero(src_mask.ravel(order='F'))
    thickness_map_3d = np.zeros(Nx * Ny * Nz)
    thickness_map_3d[ring_lin_idx] = thickness
    thickness_map_2d = thickness_map_3d.reshape((Nx, Ny, Nz), order='F')[:, :, z_src]

    h_lens_pixel = np.round(thickness_map_2d / DX_SIM).astype(int) * ring2d
    H_max = h_lens_pixel.max()
    print(f"Max lens height: {H_max} voxels ({H_max*DX_SIM*1e3:.2f} mm)")

    # Ring-shaped phase map & thickness map (Results Sec. 3.2 figure).
    phase_map_3d = np.full(Nx * Ny * Nz, np.nan)
    phase_map_3d[ring_lin_idx] = np.mod(phase_profile, 2 * np.pi)
    phase_map_2d = phase_map_3d.reshape((Nx, Ny, Nz), order='F')[:, :, z_src]
    thickness_map_2d_masked = np.where(ring2d, thickness_map_2d, np.nan)

    x_mm_ring = (np.arange(Nx) - xc) * DX_SIM * 1e3
    y_mm_ring = (np.arange(Ny) - TARGET_Y_IDX) * DX_SIM * 1e3
    extent_ring_mm = [x_mm_ring[0], x_mm_ring[-1], y_mm_ring[-1], y_mm_ring[0]]
    phase_plot = np.ma.masked_invalid(phase_map_2d)
    thick_plot = np.ma.masked_invalid(thickness_map_2d_masked * 1e3)

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    im0 = axes[0].imshow(phase_plot.T, extent=extent_ring_mm, origin='upper', cmap='twilight', vmin=0, vmax=2 * np.pi)
    axes[0].set_title(f'Ring-aperture phase map [rad] -- {medium_label}')
    plt.colorbar(im0, ax=axes[0], shrink=0.8, label='phase (rad)')
    im1 = axes[1].imshow(thick_plot.T, extent=extent_ring_mm, origin='upper', cmap='viridis')
    axes[1].set_title(f'Lens thickness map [mm] -- {medium_label}')
    plt.colorbar(im1, ax=axes[1], shrink=0.8, label='thickness (mm)')
    for ax in axes:
        ax.set_xlabel('x (mm)'); ax.set_ylabel('y (mm)'); ax.set_aspect('equal')
        for r_mm, ls in [(ID / 2 * 1e3, '--'), (OD / 2 * 1e3, '-')]:
            ax.add_patch(plt.Circle((0, 0), r_mm, fill=False, edgecolor='white', linewidth=1.2, linestyle=ls))
        for t, label in [(target_left, 'L'), (target_right, 'R')]:
            ax.plot(t['x_mm'], 0, marker='+', color='red', markersize=9, markeredgewidth=1.6)
            ax.annotate(label, (t['x_mm'], 0), color='red', fontsize=9, fontweight='bold',
                        xytext=(4, 4), textcoords='offset points')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/phase_thickness_map_{suffix}.png', dpi=160)
    plt.close(fig)

    ctx['C_LENS'], ctx['RHO_LENS'], ctx['LENS_ALPHA0'] = C_LENS, RHO_LENS, LENS_ALPHA0
    ctx['thickness_map_2d'] = thickness_map_2d
    ctx['h_lens_pixel'] = h_lens_pixel
    return ctx

### 8.4 Lens design and fabrication: STL export (Section 2.7)

In [43]:

def export_lens_stl(medium_mode, ctx):
    suffix, h_lens_pixel = ctx['suffix'], ctx['h_lens_pixel']

    from stl import mesh
    def lens_to_stl(h_lens_pixel, dx_sim, filename, base_thickness_mm=0.2):
        dx_mm = dx_sim * 1e3
        ii, jj = np.nonzero(h_lens_pixel > 0)
        n = len(ii)
        heights_mm = h_lens_pixel[ii, jj] * dx_mm
        x0 = ii * dx_mm; x1 = x0 + dx_mm
        y0 = jj * dx_mm; y1 = y0 + dx_mm
        z0 = np.full(n, -base_thickness_mm)
        z1 = heights_mm
        verts = np.stack([
            np.stack([x0, y0, z0], axis=1), np.stack([x1, y0, z0], axis=1),
            np.stack([x1, y1, z0], axis=1), np.stack([x0, y1, z0], axis=1),
            np.stack([x0, y0, z1], axis=1), np.stack([x1, y0, z1], axis=1),
            np.stack([x1, y1, z1], axis=1), np.stack([x0, y1, z1], axis=1),
        ], axis=1)
        faces = np.array([
            [0, 1, 2], [0, 2, 3], [4, 6, 5], [4, 7, 6], [0, 5, 1], [0, 4, 5],
            [1, 6, 2], [1, 5, 6], [2, 7, 3], [2, 6, 7], [3, 4, 0], [3, 7, 4],
        ])
        tris = verts[:, faces].reshape(-1, 3, 3)
        lens_mesh = mesh.Mesh(np.zeros(tris.shape[0], dtype=mesh.Mesh.dtype))
        lens_mesh.vectors[:] = tris
        lens_mesh.save(filename)
        print(f"Saved: {filename}  ({tris.shape[0]} triangles)")

    lens_to_stl(h_lens_pixel, DX_SIM, os.path.join(OUTPUT_DIR, f'phocus_lens_{suffix}.stl'), base_thickness_mm=0.2)
    return ctx

### 8.5 Forward configuration: build the combined medium (Section 2.4.2)

Voxelise the lens and build the combined acoustic medium (THE key branch point between
free-field and transcranial), plus the interactive 3-D stack figure (cf. Fig. 2.4).

In [44]:

def build_forward_medium(medium_mode, ctx):
    suffix, medium_label = ctx['suffix'], ctx['medium_label']
    thickness_map_2d = ctx['thickness_map_2d']
    C_LENS, RHO_LENS, LENS_ALPHA0 = ctx['C_LENS'], ctx['RHO_LENS'], ctx['LENS_ALPHA0']

    STANDOFF_MM = 0

    lens_thickness_vox_2d = np.where(
        np.isfinite(thickness_map_2d), np.round(thickness_map_2d / DX_SIM).astype(int), 0)
    max_thickness_vox = int(lens_thickness_vox_2d.max())
    print(f"Lens thickness range : {np.nanmin(thickness_map_2d)*1e3:.3f} - "
          f"{np.nanmax(thickness_map_2d)*1e3:.3f} mm -> up to {max_thickness_vox} voxels "
          f"({max_thickness_vox*DX_SIM*1e3:.2f} mm) axially")

    # Lens keeps 1 voxel of clearance from the transducer plane, in both media.
    LENS_Z_START = z_src + 1
    lens_mask_3d = np.zeros((Nx, Ny, Nz), dtype=bool)
    for zi in range(max_thickness_vox):
        lens_mask_3d[:, :, LENS_Z_START + zi] = lens_thickness_vox_2d > zi
    n_lens_vox = int(lens_mask_3d.sum())
    print(f"Lens voxel count : {n_lens_vox}  (~{n_lens_vox*(DX_SIM*1e3)**3:.2f} mm^3)")

    Z_SKULL_START_MM_LENS = (1 + max_thickness_vox) * DX_SIM * 1e3 + STANDOFF_MM
    print(f"Skull (re-placed{'':s}) at Z_SKULL_START_MM_LENS = {Z_SKULL_START_MM_LENS:.2f} mm "
          f"(was {Z_SKULL_START_MM:.2f} mm without the lens)")

    rescaled_intensity_domain_lens, _ = paste_skull_into_domain(
        rescaled_intensity_sim, Nx, Ny, Nz, Z_SKULL_START_MM_LENS, DX_SIM, LATERAL_OFFSET_VOX)
    density_domain_lens, speed_domain_lens, alpha_domain_lens, skull_mask_domain_lens = \
        build_acoustic_maps(rescaled_intensity_domain_lens)

    assert not np.any(lens_mask_3d & skull_mask_domain_lens), \
        "Lens and skull overlap in at least one voxel -- increase STANDOFF_MM."

    if medium_mode == "skull":
        # TRANSCRANIAL: skull as the base, lens stamped on top.
        density_full3d = density_domain_lens.copy()
        speed_full3d   = speed_domain_lens.copy()
        alpha_full3d   = alpha_domain_lens.copy()
    else:
        # FREE-FIELD: pure water everywhere, lens stamped on top. skull_mask_domain_lens is kept
        # purely as a reference for target placement / plot overlays, never merged into the medium.
        density_full3d = np.full((Nx, Ny, Nz), RHO_WATER,    dtype=np.float32)
        speed_full3d   = np.full((Nx, Ny, Nz), C_WATER,      dtype=np.float32)
        alpha_full3d   = np.full((Nx, Ny, Nz), WATER_ALPHA0, dtype=np.float32)

    density_full3d[lens_mask_3d] = RHO_LENS
    speed_full3d[lens_mask_3d]   = C_LENS
    alpha_full3d[lens_mask_3d]   = LENS_ALPHA0

    medium_lens = kWaveMedium(sound_speed=speed_full3d, density=density_full3d,
                               alpha_coeff=alpha_full3d, alpha_power=ALPHA_POWER)
    print(f"[{medium_label}] combined domain speed range : [{speed_full3d.min():.0f}, {speed_full3d.max():.0f}] m/s")

    # Sec. 2.4.2 (cont.) -- targets vs. the (re-placed/reference) skull.
    if ACCOUNT_FOR_LENS_THICKNESS:
        bregma_z_mm_lens = bregma_z_mm + Z_SKULL_START_MM_LENS
    else:
        bregma_z_mm_lens = bregma_z_mm_design + Z_SKULL_START_MM_LENS

    def shift_target_z(target, z_shift_mm, z_mm):
        new_z_mm = target['z_mm'] + z_shift_mm
        z_idx = int(np.argmin(np.abs(z_mm - new_z_mm)))
        return dict(target, z_idx=z_idx, z_mm=z_mm[z_idx])

    target_right_lens = shift_target_z(target_right, Z_SKULL_START_MM_LENS, z_mm3)
    target_left_lens  = shift_target_z(target_left,  Z_SKULL_START_MM_LENS, z_mm3)
    print(f"Left  CA1 target : z = {target_left_lens['z_mm']:.2f} mm  (was {target_left['z_mm']:.2f} mm)")
    print(f"Right CA1 target : z = {target_right_lens['z_mm']:.2f} mm  (was {target_right['z_mm']:.2f} mm)")

    def lens_bars_trace(thickness_map_2d, x_mm3, y_mm3, dx_sim, lens_z_start_mm, bar_frac=0.85,
                         color='lightsteelblue', max_bars=8000, name='Holographic lens'):
        ii, jj = np.nonzero(np.isfinite(thickness_map_2d) & (thickness_map_2d > 0))
        heights_mm = thickness_map_2d[ii, jj] * 1e3
        if len(ii) > max_bars:
            rng = np.random.default_rng(0)
            keep = rng.choice(len(ii), max_bars, replace=False)
            ii, jj, heights_mm = ii[keep], jj[keep], heights_mm[keep]
        dx_mm = dx_sim * 1e3
        half = bar_frac * dx_mm / 2.0
        n = len(ii)
        x_ctr, y_ctr = x_mm3[ii], y_mm3[jj]
        x0, x1 = x_ctr - half, x_ctr + half
        y0, y1 = y_ctr - half, y_ctr + half
        z0 = np.full(n, lens_z_start_mm)
        z1 = z0 + heights_mm
        vx = np.stack([x0, x1, x1, x0, x0, x1, x1, x0], axis=1).ravel()
        vy = np.stack([y0, y0, y1, y1, y0, y0, y1, y1], axis=1).ravel()
        vz = np.stack([z0, z0, z0, z0, z1, z1, z1, z1], axis=1).ravel()
        base_i = np.array([0, 0, 4, 4, 0, 0, 1, 1, 2, 2, 3, 3])
        base_j = np.array([1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 0, 4])
        base_k = np.array([2, 3, 6, 7, 5, 4, 6, 5, 7, 6, 4, 7])
        offsets = (np.arange(n) * 8)[:, None]
        fi = (base_i[None, :] + offsets).ravel()
        fj = (base_j[None, :] + offsets).ravel()
        fk = (base_k[None, :] + offsets).ravel()
        hover = [f"{h:.3f} mm" for h in heights_mm for _ in range(8)]
        return go.Mesh3d(x=vx, y=vy, z=vz, i=fi, j=fj, k=fk, color=color, opacity=0.95,
                          flatshading=True, text=hover, hoverinfo='text', name=name, showlegend=True)

    skull_shell_lens = skull_mask_domain_lens & ~binary_erosion(skull_mask_domain_lens, iterations=1)
    skull_idx_lens = np.argwhere(skull_shell_lens)
    src_idx_lens = np.argwhere(src_mask)

    fig3d_stack = go.Figure()
    fig3d_stack.add_trace(go.Scatter3d(
        x=x_mm3[skull_idx_lens[:, 0]], y=y_mm3[skull_idx_lens[:, 1]], z=z_mm3[skull_idx_lens[:, 2]],
        mode='markers', marker=dict(size=1.5, color='dimgray', opacity=0.15),
        name=f'Skull shell ({len(skull_idx_lens):,} pts)'))
    fig3d_stack.add_trace(go.Scatter3d(
        x=x_mm3[src_idx_lens[:, 0]], y=y_mm3[src_idx_lens[:, 1]], z=z_mm3[src_idx_lens[:, 2]],
        mode='markers', marker=dict(size=2.5, color='red', opacity=1),
        name=f'Transducer ring mask (source) ({len(src_idx_lens):,} pts)'))
    fig3d_stack.add_trace(lens_bars_trace(thickness_map_2d, x_mm3, y_mm3, DX_SIM, LENS_Z_START * DX_SIM * 1e3))
    fig3d_stack.add_trace(go.Scatter3d(
        x=[bregma_x_mm], y=[bregma_y_mm], z=[bregma_z_mm_lens],
        mode='markers+text', marker=dict(size=6, color='yellow', symbol='diamond'),
        text=['Bregma'], textposition='top center', name='Bregma', textfont=dict(color='black', size=16)))
    fig3d_stack.add_trace(go.Scatter3d(
        x=[target_left_lens['x_mm'], target_right_lens['x_mm']],
        y=[target_left_lens['y_mm'], target_right_lens['y_mm']],
        z=[target_left_lens['z_mm'], target_right_lens['z_mm']],
        mode='markers+text', marker=dict(size=7, color='lime', symbol='x'),
        text=['Left target', 'Right target'], textposition='top center',
        name='Hippocampal targets', textfont=dict(color='black', size=16)))
    fig3d_stack.update_layout(
        title=dict(text=f'Forward configuration -- {medium_label}', x=0.45, xanchor='center', y=0.99, yanchor='top', font=dict(size=30)),
        scene=dict(
            xaxis=dict(title=dict(text='x / ML (mm)', font=dict(size=20)), tickfont=dict(size=14), ticklen=15),
            yaxis=dict(title=dict(text='y / AP (mm)', font=dict(size=20)), tickfont=dict(size=14), ticklen=15),
            zaxis=dict(title=dict(text='z / depth (mm)', font=dict(size=20)), tickfont=dict(size=14), ticklen=35, autorange='reversed'),
            aspectmode='data'),
        legend=dict(itemsizing='constant', itemwidth=30, font=dict(size=16), orientation='h',
                    x=0.5, xanchor='center', y=0.93, yanchor='middle',
                    bgcolor='rgba(255,255,255,0.7)', bordercolor='rgba(0,0,0,0.2)', borderwidth=1),
        margin=dict(t=40, r=170, b=40), width=1050, height=900)
    fig3d_stack.write_html(f'{OUTPUT_DIR}/lens_stack_interactive_{suffix}.html')

    ctx['medium_lens'] = medium_lens
    ctx['speed_full3d'] = speed_full3d
    ctx['lens_mask_3d'] = lens_mask_3d
    ctx['skull_mask_domain_lens'] = skull_mask_domain_lens
    ctx['target_left_lens'] = target_left_lens
    ctx['target_right_lens'] = target_right_lens
    ctx['Z_SKULL_START_MM_LENS'] = Z_SKULL_START_MM_LENS
    return ctx

### 8.6 Lens-verification simulation (Section 2.4.2)

In [45]:

def run_lens_verification_sim(medium_mode, ctx):
    suffix, medium_label = ctx['suffix'], ctx['medium_label']
    medium_lens, speed_full3d = ctx['medium_lens'], ctx['speed_full3d']

    C_MAX_LENS_RUN = float(max(C_WATER, speed_full3d.max()))
    kgrid_lens = kWaveGrid([Nx, Ny, Nz], [DX_SIM, DX_SIM, DX_SIM])
    if medium_mode == "skull":
        kgrid_lens.makeTime(C_MAX_LENS_RUN, cfl=0.1, t_end=T_END)
    else:
        kgrid_lens.makeTime(C_MAX_LENS_RUN, cfl=0.2, t_end=T_END)
    print(f"[{medium_label}] C_MAX_LENS_RUN = {C_MAX_LENS_RUN:.0f} m/s   dt = {kgrid_lens.dt*1e9:.2f} ns   Nt = {kgrid_lens.Nt}")

    sig_lens = generate_source_signal(SOURCE_WAVEFORM, 1 / kgrid_lens.dt, F0, N_CYCLES, P0, t_array=kgrid_lens.t_array)
    input_signal_lens = np.tile(sig_lens, (n_src, 1)).astype(np.float32)

    source_lens = kSource()
    source_lens.p_mask = src_mask
    source_lens.p      = input_signal_lens

    sensor_lens = kSensor()
    sensor_lens.mask   = np.ones((Nx, Ny, Nz), dtype=bool)
    sensor_lens.record = ['p_max', 'p_min']

    sim_opts_lens = SimulationOptions(
        input_filename  = os.path.join(OUTPUT_DIR, f'in_lens_{suffix}.h5'),
        output_filename = os.path.join(OUTPUT_DIR, f'out_lens_{suffix}.h5'),
        **sim_opts_common)

    print(f"[{medium_label}] Running the lens-verification simulation...")
    out_lens = kspaceFirstOrder3DC(kgrid=kgrid_lens, medium=medium_lens, source=source_lens,
                                    sensor=sensor_lens, simulation_options=sim_opts_lens,
                                    execution_options=exec_opts)

    p_max_lens = out_lens['p_max'].reshape((Nx, Ny, Nz), order='F')
    p_min_lens = out_lens['p_min'].reshape((Nx, Ny, Nz), order='F')
    PNP_lens   = np.abs(p_min_lens)
    print(f"Global peak positive pressure : {p_max_lens.max()/1e3:.1f} kPa")
    print(f"Global peak negative pressure : {PNP_lens.max()/1e3:.1f} kPa  (PNP)")

    ctx['PNP_lens'] = PNP_lens
    return ctx

### 8.7 Bilateral focal-performance metrics (Section 3.3)

In [46]:

def compute_focal_metrics(medium_mode, ctx):
    medium_label = ctx['medium_label']
    PNP_lens = ctx['PNP_lens']
    target_left_lens, target_right_lens = ctx['target_left_lens'], ctx['target_right_lens']

    m_lens_L = per_focus_metrics(PNP_lens, target_left_lens['x_idx'],  target_left_lens['z_idx'],  TARGET_Y_IDX, f"LEFT focus  ({medium_label})")
    m_lens_R = per_focus_metrics(PNP_lens, target_right_lens['x_idx'], target_right_lens['z_idx'], TARGET_Y_IDX, f"RIGHT focus ({medium_label})")

    two_distinct_peaks = (abs(target_left_lens['x_idx'] - target_right_lens['x_idx']) >= 2)
    interfocal_lateral_mm = abs(m_lens_R['x_mm'] - m_lens_L['x_mm'])
    interfocal_3d_mm = np.sqrt((m_lens_R['x_mm'] - m_lens_L['x_mm'])**2 + (m_lens_R['z_mm'] - m_lens_L['z_mm'])**2)
    p_hi, p_lo = max(m_lens_L['peak_kPa'], m_lens_R['peak_kPa']), min(m_lens_L['peak_kPa'], m_lens_R['peak_kPa'])
    uniformity_dB = 20 * np.log10(p_hi / p_lo) if p_lo > 0 else np.nan

    xz_slice = PNP_lens[:, TARGET_Y_IDX, :].copy()
    EXCLUDE_MM = 1.5
    XX, ZZ = np.meshgrid(x_mm, z_mm, indexing='ij')
    excl = (np.hypot(XX - m_lens_L['x_mm'], ZZ - m_lens_L['z_mm']) < EXCLUDE_MM) | \
           (np.hypot(XX - m_lens_R['x_mm'], ZZ - m_lens_R['z_mm']) < EXCLUDE_MM)
    xz_slice[excl] = 0
    gl_idx = np.unravel_index(np.argmax(xz_slice), xz_slice.shape)
    grating_lobe_kPa = xz_slice[gl_idx] / 1e3
    grating_lobe_dB = 20 * np.log10(grating_lobe_kPa / p_lo) if p_lo > 0 else np.nan
    print(f"Inter-focal distance (lateral): {interfocal_lateral_mm:.2f} mm | Uniformity: {uniformity_dB:.2f} dB | "
          f"Grating lobe: {grating_lobe_kPa:.1f} kPa ({grating_lobe_dB:.1f} dB)")

    ctx['m_lens_L'], ctx['m_lens_R'] = m_lens_L, m_lens_R
    ctx['interfocal_lateral_mm'] = interfocal_lateral_mm
    ctx['uniformity_dB'] = uniformity_dB
    ctx['two_distinct_peaks'] = two_distinct_peaks
    ctx['grating_lobe_kPa'] = grating_lobe_kPa
    ctx['grating_lobe_dB'] = grating_lobe_dB
    return ctx

### 8.8 Full-domain pressure-field figure (Section 3.3)

XZ on-axis + XY at target depth + axial profile (not the normalized/cropped view -- that's
the He2023-style figure further below).

In [47]:

def plot_pressure_field_figure(medium_mode, ctx):
    suffix, medium_label, RUN_TAG_PRETTY = ctx['suffix'], ctx['medium_label'], ctx['RUN_TAG_PRETTY']
    PNP_lens = ctx['PNP_lens']
    target_left_lens = ctx['target_left_lens']
    skull_mask_domain_lens, lens_mask_3d = ctx['skull_mask_domain_lens'], ctx['lens_mask_3d']
    m_lens_L, m_lens_R = ctx['m_lens_L'], ctx['m_lens_R']

    TARGET_Z_IDX_LENS = target_left_lens['z_idx']
    fig, axes = plt.subplots(1, 3, figsize=(19, 6))
    VMAX_FIELD = float(PNP_lens.max())
    show_sim_slice(PNP_lens, 'y', index=TARGET_Y_IDX, ax=axes[0], cmap='hot',
                   overlay_mask=skull_mask_domain_lens, vmin=0, vmax=VMAX_FIELD,
                   title=f'PNP -- XZ on-axis ({medium_label}, {RUN_TAG_PRETTY})')
    axes[0].contour(x_mm, z_mm, lens_mask_3d[:, TARGET_Y_IDX, :].T, levels=[0.5], colors='deepskyblue', linewidths=1.0)
    for m, lbl in [(m_lens_L, 'L'), (m_lens_R, 'R')]:
        axes[0].plot(m['x_mm'], m['z_mm'], marker='x', color='lime', markersize=10, markeredgewidth=2)
        axes[0].annotate(lbl, (m['x_mm'], m['z_mm']), color='lime', fontsize=10, fontweight='bold', xytext=(4, 4), textcoords='offset points')

    show_sim_slice(PNP_lens, 'z', index=TARGET_Z_IDX_LENS, ax=axes[1], cmap='hot',
                   overlay_mask=skull_mask_domain_lens, vmin=0, vmax=VMAX_FIELD,
                   title=f"PNP -- XY at target depth (z = {z_mm[TARGET_Z_IDX_LENS]:.2f} mm)")
    for m, lbl in [(m_lens_L, 'L'), (m_lens_R, 'R')]:
        axes[1].plot(m['x_mm'], y_mm[TARGET_Y_IDX], marker='x', color='lime', markersize=10, markeredgewidth=2)
        axes[1].annotate(lbl, (m['x_mm'], y_mm[TARGET_Y_IDX]), color='lime', fontsize=10, fontweight='bold', xytext=(4, 4), textcoords='offset points')

    axial_lens = PNP_lens[xc, TARGET_Y_IDX, :]
    axes[2].plot(z_mm, axial_lens / 1e3, lw=1.8, color='tab:purple')
    axes[2].axvline(m_lens_L['z_mm'], color='tab:blue', ls='--', lw=1, label=f"L focus z={m_lens_L['z_mm']:.2f} mm")
    axes[2].axvline(m_lens_R['z_mm'], color='tab:orange', ls='--', lw=1, label=f"R focus z={m_lens_R['z_mm']:.2f} mm")
    axes[2].set_xlabel('z (mm)'); axes[2].set_ylabel('|p| (kPa)')
    axes[2].set_title(f'On-axis (x=0) profile -- {medium_label}')
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
    plt.suptitle(f'Pressure field with the physical holographic lens ({medium_label})')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/pressure_field_{suffix}.png', dpi=160)
    plt.close(fig)

    ctx['TARGET_Z_IDX_LENS'] = TARGET_Z_IDX_LENS
    return ctx

### 8.9 Lateral profile through both foci & focal-metrics summary (Section 3.3)

In [48]:

def plot_lateral_profile_and_summary(medium_mode, ctx):
    suffix, medium_label, RUN_TAG_PRETTY = ctx['suffix'], ctx['medium_label'], ctx['RUN_TAG_PRETTY']
    PNP_lens, TARGET_Z_IDX_LENS = ctx['PNP_lens'], ctx['TARGET_Z_IDX_LENS']
    m_lens_L, m_lens_R = ctx['m_lens_L'], ctx['m_lens_R']
    interfocal_lateral_mm, uniformity_dB = ctx['interfocal_lateral_mm'], ctx['uniformity_dB']
    two_distinct_peaks = ctx['two_distinct_peaks']
    grating_lobe_kPa, grating_lobe_dB = ctx['grating_lobe_kPa'], ctx['grating_lobe_dB']

    lateral_at_target = PNP_lens[:, TARGET_Y_IDX, TARGET_Z_IDX_LENS]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(x_mm, lateral_at_target / 1e3, lw=2.0, color='tab:purple',
            label=f'x-profile @ target depth z={z_mm[TARGET_Z_IDX_LENS]:.2f} mm')
    for m, colour in [(m_lens_L, 'tab:blue'), (m_lens_R, 'tab:orange')]:
        prof = PNP_lens[:, TARGET_Y_IDX, m['z_idx']]
        _, x_lo_mm, x_hi_mm = fwhm(prof, x_mm)
        ax.axvspan(x_lo_mm, x_hi_mm, color=colour, alpha=0.15)
        ax.plot(m['x_mm'], m['peak_kPa'], marker='x', color=colour, markersize=11, markeredgewidth=2.2)
    ax.set_xlabel('x (mm)'); ax.set_ylabel('|p| (kPa)')
    ax.set_title(f'Lateral pressure profile through both targets ({medium_label}) -- {RUN_TAG_PRETTY}\n'
                 f'Inter-focal distance = {interfocal_lateral_mm:.2f} mm   |   Uniformity = {uniformity_dB:.2f} dB')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/lateral_pressure_profile_{suffix}.png', dpi=160)
    plt.close(fig)

    intended_separation_mm = abs(target_right['x_mm'] - target_left['x_mm'])

    _summary_path = os.path.join(OUTPUT_DIR, f"focal_metrics_summary_{suffix}.txt")
    with open(_summary_path, "w") as _f:
        _f.write("=" * 70 + "\n")
        _f.write(f"LENS-VERIFICATION FOCAL METRICS SUMMARY -- {medium_label}\n")
        _f.write("=" * 70 + "\n")
        _f.write(f"RUN_TAG                          : {RUN_TAG}\n")
        _f.write(f"Two distinct peaks resolved      : {two_distinct_peaks}\n")
        _f.write(f"L focus: PNP={m_lens_L['peak_kPa']:.1f} kPa, FWHM_ax={m_lens_L['fwhm_ax_mm']:.2f} mm, FWHM_lat={m_lens_L['fwhm_lat_mm']:.2f} mm\n")
        _f.write(f"R focus: PNP={m_lens_R['peak_kPa']:.1f} kPa, FWHM_ax={m_lens_R['fwhm_ax_mm']:.2f} mm, FWHM_lat={m_lens_R['fwhm_lat_mm']:.2f} mm\n")
        _f.write(f"Inter-focal distance (lateral)   : {interfocal_lateral_mm:.2f} mm (intended separation = {intended_separation_mm:.2f} mm)\n")
        _f.write(f"Pressure uniformity between foci : {uniformity_dB:.2f} dB\n")
        _f.write(f"Highest grating/side lobe        : {grating_lobe_kPa:.1f} kPa  ({grating_lobe_dB:.1f} dB rel. weaker focus)\n")
    print(f"Focal metrics summary saved -> {_summary_path}")
    return ctx

### 8.10 Normalized 2x2 summary figure (Sections 3.2, 3.3)

XZ plane / XY plane / axial profile / lateral profile, cropped to the post-lens depth
window and normalized to the peak WITHIN that window -- the He2023-style figure, shared by
both media.

In [49]:

def plot_normalized_summary_figure(medium_mode, ctx):
    suffix = ctx['suffix']
    PNP_lens = ctx['PNP_lens']
    target_left_lens, target_right_lens = ctx['target_left_lens'], ctx['target_right_lens']
    skull_mask_domain_lens, lens_mask_3d = ctx['skull_mask_domain_lens'], ctx['lens_mask_3d']
    Z_SKULL_START_MM_LENS = ctx['Z_SKULL_START_MM_LENS']

    Z_ZOOM_MIN, Z_ZOOM_MAX = Z_SKULL_START_MM_LENS, 15.0
    z_zoom_mask = (z_mm >= Z_ZOOM_MIN) & (z_mm <= Z_ZOOM_MAX)
    p_zoom_slab = PNP_lens[:, TARGET_Y_IDX, z_zoom_mask]
    p_norm = PNP_lens / p_zoom_slab.max()

    TITLE_FS, LABEL_FS, TICK_FS, LEGEND_FS, SUPTITLE_FS = 16, 14, 12, 11, 19

    fig, axes = plt.subplots(2, 2, figsize=(13, 11))

    show_sim_slice(p_norm, 'y', index=TARGET_Y_IDX, ax=axes[0, 0], cmap='jet',
                   overlay_mask=skull_mask_domain_lens, vmin=0, vmax=1, title='XZ Plane')
    axes[0, 0].contour(x_mm, z_mm, lens_mask_3d[:, TARGET_Y_IDX, :].T, levels=[0.5], colors='deepskyblue', linewidths=1.0)
    for t, lbl, mcolor in [(target_left_lens, 'L', 'black'), (target_right_lens, 'R', 'white')]:
        axes[0, 0].plot(t['x_mm'], t['z_mm'], marker='+', color=mcolor, markersize=10, markeredgewidth=1.8)
        axes[0, 0].annotate(lbl, (t['x_mm'], t['z_mm']), color=mcolor, fontsize=10, fontweight='bold', xytext=(4, 4), textcoords='offset points')
    _y0, _y1 = axes[0, 0].get_ylim()
    axes[0, 0].set_ylim((Z_ZOOM_MAX, Z_ZOOM_MIN) if _y0 > _y1 else (Z_ZOOM_MIN, Z_ZOOM_MAX))

    TARGET_Z_IDX = target_left_lens['z_idx']
    show_sim_slice(p_norm, 'z', index=TARGET_Z_IDX, ax=axes[0, 1], cmap='jet', vmin=0, vmax=1,
                   title=f'XY Plane at z = {z_mm[TARGET_Z_IDX]:.2f} mm')

    axial_raw = PNP_lens[xc, TARGET_Y_IDX, :]
    axial_norm = axial_raw / axial_raw[z_zoom_mask].max()
    axes[1, 0].plot(z_mm, axial_norm, lw=1.8, color='tab:purple')
    axes[1, 0].axvline(target_left_lens['z_mm'], color='tab:blue', ls='--', lw=1, label=f"L target z={target_left_lens['z_mm']:.2f} mm")
    axes[1, 0].axvline(target_right_lens['z_mm'], color='tab:orange', ls='--', lw=1, label=f"R target z={target_right_lens['z_mm']:.2f} mm")
    axes[1, 0].set_xlabel('z (mm)'); axes[1, 0].set_ylabel('Normalized PNP')
    axes[1, 0].set_title('Axial Profile (x = 0)')
    axes[1, 0].legend(fontsize=LEGEND_FS); axes[1, 0].grid(alpha=0.3); axes[1, 0].set_ylim(0, 1.05)
    axes[1, 0].set_xlim(Z_ZOOM_MIN, Z_ZOOM_MAX)

    x_profile = p_norm[:, TARGET_Y_IDX, TARGET_Z_IDX]
    axes[1, 1].plot(x_mm, x_profile, lw=1.8, color='tab:red')
    axes[1, 1].axvline(target_left_lens['x_mm'], color='tab:blue', ls='--', lw=1, label=f"L target x={target_left_lens['x_mm']:+.2f} mm")
    axes[1, 1].axvline(target_right_lens['x_mm'], color='tab:orange', ls='--', lw=1, label=f"R target x={target_right_lens['x_mm']:+.2f} mm")
    axes[1, 1].set_xlabel('x (mm)'); axes[1, 1].set_ylabel('Normalized PNP')
    axes[1, 1].set_title('Lateral Profile at Target Depth')
    axes[1, 1].legend(fontsize=LEGEND_FS); axes[1, 1].grid(alpha=0.3); axes[1, 1].set_ylim(0, 1.05)

    for ax in axes.flat:
        ax.title.set_fontsize(TITLE_FS)
        ax.xaxis.label.set_size(LABEL_FS)
        ax.yaxis.label.set_size(LABEL_FS)
        ax.tick_params(axis='both', labelsize=TICK_FS)

    suptitle_text = 'Pressure Distribution in Free Field' if medium_mode == "free_field" \
        else 'Pressure Distribution — Transcranial (Skull + Lens)'
    plt.suptitle(suptitle_text, fontsize=SUPTITLE_FS, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f'{OUTPUT_DIR}/normalized_pressure_field_{suffix}.png', dpi=160)
    plt.close(fig)
    return ctx

### 8.11 Pipeline orchestration: `run_lens_pipeline(medium_mode)`

Chains the section functions above in order, threading their outputs through the shared
`ctx` dict, for one medium.

In [50]:

def run_lens_pipeline(medium_mode: str):
    """medium_mode: 'free_field' or 'skull'. Runs the full per-medium pipeline -- virtual-source
    phase extraction, lens design, lens build, verification simulation, metrics and figures --
    for that medium. All artifacts land in the shared OUTPUT_DIR with a _freefield / _skull
    filename suffix."""
    assert medium_mode in ("free_field", "skull")
    suffix = "freefield" if medium_mode == "free_field" else "skull"
    medium_label = "free field" if medium_mode == "free_field" else "transcranial (skull + lens)"
    ctx = dict(suffix=suffix, medium_label=medium_label, RUN_TAG_PRETTY=run_tag_pretty(medium_label))

    ctx = run_phase_extraction_sim(medium_mode, ctx)
    ctx = extract_phase(medium_mode, ctx)
    ctx = design_lens_and_thickness_map(medium_mode, ctx)
    ctx = export_lens_stl(medium_mode, ctx)
    ctx = build_forward_medium(medium_mode, ctx)
    ctx = run_lens_verification_sim(medium_mode, ctx)
    ctx = compute_focal_metrics(medium_mode, ctx)
    ctx = plot_pressure_field_figure(medium_mode, ctx)
    ctx = plot_lateral_profile_and_summary(medium_mode, ctx)
    ctx = plot_normalized_summary_figure(medium_mode, ctx)

    print(f"[{medium_label}] Done -- figures saved to {OUTPUT_DIR} with suffix '_{suffix}'.")
    return dict(
        PNP_lens=ctx['PNP_lens'], m_lens_L=ctx['m_lens_L'], m_lens_R=ctx['m_lens_R'],
        interfocal_lateral_mm=ctx['interfocal_lateral_mm'], uniformity_dB=ctx['uniformity_dB'],
        thickness_map_2d=ctx['thickness_map_2d'],
    )

## 9. Run the pipeline: select medium(s) via `medium_mode`

This is the "one variable, several options" switch for the whole notebook: set/loop
`medium_mode` over `"free_field"` and/or `"skull"` to choose which of the two Sec. 2.4.2
simulation configurations is executed. As written below it runs BOTH media in one pass, so
both lens designs and both verification runs land in the same `OUTPUT_DIR` (Sec. 2.4-2.7 and
Ch. 3 results for each medium), with every filename suffixed `_freefield` / `_skull`.

In [51]:

if __name__ == "__main__":
    results = {}
    for medium_mode in ("free_field", "skull"):
        print(f"\n{'=' * 70}\nRunning pipeline for medium_mode={medium_mode!r}\n{'=' * 70}")
        results[medium_mode] = run_lens_pipeline(medium_mode)

    print(f"\nAll done. Both media's figures are in: {OUTPUT_DIR}")
    print("  *_freefield.*  = free-field (water + lens)")
    print("  *_skull.*      = transcranial (skull + lens)")


Running pipeline for medium_mode='free_field'



[free field] Running the virtual-source phase-extraction simulation...


c:\Users\lucia\anaconda3\Lib\site-packages\kwave\options\simulation_execution_options.py:111: UserWarning: Custom binary name set. Ignoring `is_gpu_simulation` state.
  warnings.warn("Custom binary name set. Ignoring `is_gpu_simulation` state.")


+---------------------------------------------------------------+
|                   kspaceFirstOrder-OMP v1.3                   |
+---------------------------------------------------------------+
| Reading simulation configuration:                        Done |
| Number of CPU threads:                                     16 |
| Processor name:                Intel(R) Core(TM) Ultra 7 255H |
+---------------------------------------------------------------+
|                      Simulation details                       |
+---------------------------------------------------------------+
| Domain dimensions:                            199 x 260 x 227 |
| Medium type:                                               3D |
| Simulation time steps:                                   2642 |
+---------------------------------------------------------------+
| Input file:  c:\Users\lucia\OneDrive - Imperial College       |
|              London\msc project\Kwave simulations\phase_      |
|         

[free field] Running the lens-verification simulation...


c:\Users\lucia\anaconda3\Lib\site-packages\kwave\options\simulation_execution_options.py:111: UserWarning:

Custom binary name set. Ignoring `is_gpu_simulation` state.



+---------------------------------------------------------------+
|                   kspaceFirstOrder-OMP v1.3                   |
+---------------------------------------------------------------+
| Reading simulation configuration:                        Done |
| Number of CPU threads:                                     16 |
| Processor name:                Intel(R) Core(TM) Ultra 7 255H |
+---------------------------------------------------------------+
|                      Simulation details                       |
+---------------------------------------------------------------+
| Domain dimensions:                            199 x 260 x 227 |
| Medium type:                                               3D |
| Simulation time steps:                                   6931 |
+---------------------------------------------------------------+
| Input file:  c:\Users\lucia\OneDrive - Imperial College       |
|              London\msc project\Kwave simulations\phase_      |
|         

+---------------------------------------------------------------+
|                   kspaceFirstOrder-OMP v1.3                   |
+---------------------------------------------------------------+
| Reading simulation configuration:                        Done |
| Number of CPU threads:                                     16 |
| Processor name:                Intel(R) Core(TM) Ultra 7 255H |
+---------------------------------------------------------------+
|                      Simulation details                       |
+---------------------------------------------------------------+
| Domain dimensions:                            199 x 260 x 227 |
| Medium type:                                               3D |
| Simulation time steps:                                  12446 |
+---------------------------------------------------------------+
| Input file:  c:\Users\lucia\OneDrive - Imperial College       |
|              London\msc project\Kwave simulations\phase_      |
|         

[transcranial (skull + lens)] Running the lens-verification simulation...


+---------------------------------------------------------------+
|                   kspaceFirstOrder-OMP v1.3                   |
+---------------------------------------------------------------+
| Reading simulation configuration:                        Done |
| Number of CPU threads:                                     16 |
| Processor name:                Intel(R) Core(TM) Ultra 7 255H |
+---------------------------------------------------------------+
|                      Simulation details                       |
+---------------------------------------------------------------+
| Domain dimensions:                            199 x 260 x 227 |
| Medium type:                                               3D |
| Simulation time steps:                                  13862 |
+---------------------------------------------------------------+
| Input file:  c:\Users\lucia\OneDrive - Imperial College       |
|              London\msc project\Kwave simulations\phase_      |
|         